# Secretory Lineage scRNA-seq BBKNN Analysis with Marker Identification

**Purpose**: BBKNN-based reclustering and comprehensive marker gene analysis

**Key Features**:
1. Small batch filtering (following QUICK_REFERENCE_MEMORY best practices)
2. BBKNN clustering with multiple resolutions
3. Marker gene identification per cluster and CellTypist annotation
4. Comprehensive visualization

**Input**: Secretory_Lineage_filtered.h5ad

**Author**: r2end
**Date**: 2025-01-16
**Version**: v1.0



In [ ]:
# ===== Configuration =====
import warnings
warnings.filterwarnings('ignore')

import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import gc
from scipy import sparse
from typing import Optional, Dict, List, Tuple
import logging

# Check bbknn availability
try:
    import bbknn
    print(f"✓ bbknn version: {bbknn.__version__ if hasattr(bbknn, '__version__') else 'unknown'}")
except ImportError:
    raise ImportError("bbknn not installed. Install: pip install bbknn")

# Set plotting parameters
sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi=100, facecolor='white', frameon=False)
sc.set_figure_params(scanpy=True, dpi=100, dpi_save=300, 
                      vector_friendly=True, fontsize=12)
sc.settings.n_jobs = 48  # Utilize multi-core

# File paths
INPUT_FILE = '/home/h2048/data/py/0114/cnmf_v1.3_fixed_optimized_k/Secretory_Lineage_filtered.h5ad'
OUTPUT_DIR = '/home/h2048/data/py/0114/cnmf_v1.3_fixed_optimized_k/bbknn_analysis/'

# Create output directory
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Random seed for reproducibility
np.random.seed(42)

# BBKNN parameters
BBKNN_CONFIG = {
    'batch_key': None,  # Will auto-detect
    'n_pcs': 50,
    'neighbors_within_batch': 3,
    'metric': 'euclidean',
    'trim': None,
    'min_cells_per_batch': 10,  # Filter small batches
    'min_batches': 2,
    'leiden_resolutions': [0.5, 1.0, 1.5],
    'default_resolution': 1.0,
    'umap_min_dist': 0.3,
    'umap_spread': 1.0,
}

# Marker analysis parameters
MARKER_CONFIG = {
    'min_pct': 0.25,
    'logfc_threshold': 0.25,
    'top_n': 20,
    'method': 'wilcoxon',  # Fast for exploratory
    'use_raw': True,
}

print(f"\n{'='*80}")
print("SECRETORY LINEAGE BBKNN ANALYSIS")
print(f"{'='*80}\n")



## 1. Data Loading and Structure Inspection



In [ ]:
print(f"{'='*80}")
print("STEP 1: DATA LOADING")
print(f"{'='*80}\n")

# Load data
adata = sc.read_h5ad(INPUT_FILE)

print(f"✓ Data loaded successfully")
print(f"  Shape: {adata.shape[0]:,} cells × {adata.shape[1]:,} genes")

# Memory usage
if sparse.issparse(adata.X):
    mem_gb = adata.X.data.nbytes / 1e9
    print(f"  Memory: {mem_gb:.2f} GB (sparse)")
else:
    mem_gb = adata.X.nbytes / 1e9
    print(f"  Memory: {mem_gb:.2f} GB (dense)")

# Check data structure
print(f"\nData structure:")
print(f"  .X type: {type(adata.X).__name__}")
print(f"  .layers: {list(adata.layers.keys())}")
print(f"  .obsm: {list(adata.obsm.keys())}")
print(f"  .raw: {'Available' if adata.raw else 'Not available'}")

# Check required layer
if 'counts' not in adata.layers and 'log1p' not in adata.layers:
    raise ValueError("Missing required layer: 'counts' or 'log1p'")

print(f"\n✓ Data structure verified")



## 2. Batch Key Detection and QC



In [ ]:
print(f"\n{'='*80}")
print("STEP 2: BATCH KEY DETECTION")
print(f"{'='*80}\n")

# Auto-detect batch key
batch_key_candidates = ['dataset', 'sample', 'sample_id', 'orig.ident']
batch_key = None

for candidate in batch_key_candidates:
    if candidate in adata.obs.columns:
        n_batches = adata.obs[candidate].nunique()
        if n_batches >= BBKNN_CONFIG['min_batches']:
            batch_key = candidate
            print(f"✓ Detected batch key: '{batch_key}' ({n_batches} batches)")
            break

if batch_key is None:
    raise ValueError(f"No valid batch key found. Tried: {batch_key_candidates}")

BBKNN_CONFIG['batch_key'] = batch_key

# Batch composition analysis
batch_counts = adata.obs[batch_key].value_counts().sort_values(ascending=False)
print(f"\nBatch composition:")
print(f"  Total batches: {len(batch_counts)}")
print(f"  Cells per batch (top 10):")
for batch, count in batch_counts.head(10).items():
    print(f"    {batch}: {count:,} cells")
if len(batch_counts) > 10:
    print(f"    ... and {len(batch_counts) - 10} more batches")



## 3. Small Batch Filtering (CRITICAL)



In [ ]:
print(f"\n{'='*80}")
print("STEP 3: SMALL BATCH FILTERING")
print(f"{'='*80}\n")

# Identify small batches
small_batches = batch_counts[batch_counts < BBKNN_CONFIG['min_cells_per_batch']]

if len(small_batches) > 0:
    print(f"⚠️  Found {len(small_batches)} small batches (< {BBKNN_CONFIG['min_cells_per_batch']} cells):")
    for batch, count in small_batches.items():
        print(f"    {batch}: {count} cells")
    
    # Filter
    valid_batches = batch_counts[batch_counts >= BBKNN_CONFIG['min_cells_per_batch']].index
    n_valid = len(valid_batches)
    
    if n_valid < BBKNN_CONFIG['min_batches']:
        raise ValueError(f"Too few valid batches after filtering ({n_valid} < {BBKNN_CONFIG['min_batches']})")
    
    print(f"\n  Filtering to {n_valid} valid batches...")
    adata = adata[adata.obs[batch_key].isin(valid_batches)].copy()
    print(f"  Retained: {adata.n_obs:,} cells")
    
    # Update batch counts
    batch_counts = adata.obs[batch_key].value_counts()
else:
    print(f"✓ All batches have sufficient cells (≥{BBKNN_CONFIG['min_cells_per_batch']})")

# Auto-adjust neighbors_within_batch
min_batch_size = batch_counts.min()
neighbors_within = BBKNN_CONFIG['neighbors_within_batch']

if neighbors_within >= min_batch_size:
    original_neighbors = neighbors_within
    neighbors_within = max(1, min_batch_size - 1)
    print(f"\n⚠️  Auto-adjusting neighbors_within_batch:")
    print(f"    {original_neighbors} → {neighbors_within} (min batch size: {min_batch_size})")
    BBKNN_CONFIG['neighbors_within_batch'] = neighbors_within



## 4. Backup Original Embeddings



In [ ]:
print(f"\n{'='*80}")
print("STEP 4: BACKUP ORIGINAL EMBEDDINGS")
print(f"{'='*80}\n")

# Backup existing UMAP if present
if 'X_umap' in adata.obsm:
    if 'X_umap_original' not in adata.obsm:
        adata.obsm['X_umap_original'] = np.asarray(adata.obsm['X_umap']).copy()
        print(f"✓ Backed up X_umap → X_umap_original")
    else:
        print(f"  X_umap_original already exists, skipping backup")

# Backup existing leiden if present
if 'leiden' in adata.obs.columns:
    if 'leiden_original' not in adata.obs.columns:
        adata.obs['leiden_original'] = adata.obs['leiden'].copy()
        print(f"✓ Backed up leiden → leiden_original")



## 5. PCA Computation (if needed)



In [ ]:
print(f"\n{'='*80}")
print("STEP 5: PCA COMPUTATION")
print(f"{'='*80}\n")

need_pca = (
    'X_pca' not in adata.obsm or 
    adata.obsm['X_pca'].shape[1] < BBKNN_CONFIG['n_pcs']
)

if need_pca:
    print(f"  Computing PCA with {BBKNN_CONFIG['n_pcs']} components...")
    
    # Use HVG if available
    use_hvg = (
        'highly_variable' in adata.var.columns and 
        np.any(adata.var['highly_variable'].values)
    )
    
    if use_hvg:
        n_hvg = adata.var['highly_variable'].sum()
        print(f"    Using {n_hvg:,} highly variable genes")
    
    try:
        sc.pp.pca(
            adata, 
            n_comps=BBKNN_CONFIG['n_pcs'], 
            svd_solver='arpack',
            use_highly_variable=use_hvg,
            random_state=42
        )
        print(f"✓ PCA computed")
    except Exception as e:
        raise RuntimeError(f"PCA failed: {e}")
else:
    print(f"✓ PCA already available ({adata.obsm['X_pca'].shape[1]} components)")



## 6. BBKNN Graph Construction



In [ ]:
print(f"\n{'='*80}")
print("STEP 6: BBKNN GRAPH CONSTRUCTION")
print(f"{'='*80}\n")

print(f"  Parameters:")
print(f"    batch_key: {BBKNN_CONFIG['batch_key']}")
print(f"    neighbors_within_batch: {BBKNN_CONFIG['neighbors_within_batch']}")
print(f"    n_pcs: {BBKNN_CONFIG['n_pcs']}")
print(f"    metric: {BBKNN_CONFIG['metric']}")

try:
    bbknn.bbknn(
        adata,
        batch_key=BBKNN_CONFIG['batch_key'],
        neighbors_within_batch=BBKNN_CONFIG['neighbors_within_batch'],
        n_pcs=BBKNN_CONFIG['n_pcs'],
        metric=BBKNN_CONFIG['metric'],
        trim=BBKNN_CONFIG['trim'],
        key_added='neighbors_bbknn',
        copy=False,
    )
    print(f"\n✓ BBKNN graph constructed successfully")
except Exception as e:
    raise RuntimeError(f"BBKNN failed: {e}")



## 7. UMAP Computation



In [ ]:
print(f"\n{'='*80}")
print("STEP 7: UMAP COMPUTATION")
print(f"{'='*80}\n")

try:
    sc.tl.umap(
        adata,
        neighbors_key='neighbors_bbknn',
        min_dist=BBKNN_CONFIG['umap_min_dist'],
        spread=BBKNN_CONFIG['umap_spread'],
        random_state=42,
    )
    print(f"✓ UMAP computed")
except Exception as e:
    raise RuntimeError(f"UMAP failed: {e}")

# Rename to distinguish from original
adata.obsm['X_umap_bbknn'] = adata.obsm['X_umap'].copy()



## 8. Leiden Clustering at Multiple Resolutions



In [ ]:
print(f"\n{'='*80}")
print("STEP 8: LEIDEN CLUSTERING")
print(f"{'='*80}\n")

for res in BBKNN_CONFIG['leiden_resolutions']:
    key = f'leiden_bbknn_res{res}'
    
    print(f"  Computing leiden at resolution {res}...")
    
    try:
        sc.tl.leiden(
            adata,
            neighbors_key='neighbors_bbknn',
            resolution=res,
            key_added=key,
            flavor='igraph',
            n_iterations=2,
            directed=False,
            random_state=42
        )
    except TypeError:
        # Fallback if flavor not supported
        sc.tl.leiden(
            adata,
            neighbors_key='neighbors_bbknn',
            resolution=res,
            key_added=key,
            n_iterations=2,
            random_state=42
        )
    
    n_clusters = adata.obs[key].nunique()
    print(f"    → {n_clusters} clusters")

# Set default leiden
default_key = f'leiden_bbknn_res{BBKNN_CONFIG["default_resolution"]}'
adata.obs['leiden_bbknn'] = adata.obs[default_key].copy()

print(f"\n✓ Clustering completed")
print(f"  Default clustering: leiden_bbknn ({adata.obs['leiden_bbknn'].nunique()} clusters)")



## 9. UMAP Visualization - Batch Integration Check



In [ ]:
print(f"\n{'='*80}")
print("STEP 9: BATCH INTEGRATION VISUALIZATION")
print(f"{'='*80}\n")

# Batch integration
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Before BBKNN (if available)
if 'X_umap_original' in adata.obsm:
    sc.pl.embedding(
        adata, basis='umap_original', color=batch_key,
        ax=axes[0], show=False, title='Before BBKNN'
    )
else:
    axes[0].text(0.5, 0.5, 'Original UMAP not available',
                ha='center', va='center', fontsize=14)
    axes[0].axis('off')

# After BBKNN
sc.pl.embedding(
    adata, basis='umap_bbknn', color=batch_key,
    ax=axes[1], show=False, title='After BBKNN'
)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}batch_integration_comparison.pdf', bbox_inches='tight', dpi=300)
plt.show()

print(f"✓ Batch integration plot saved")



## 10. UMAP Visualization - Clustering Results



In [ ]:
print(f"\n{'='*80}")
print("STEP 10: CLUSTERING VISUALIZATION")
print(f"{'='*80}\n")

# Plot all leiden resolutions
leiden_keys = [f'leiden_bbknn_res{res}' for res in BBKNN_CONFIG['leiden_resolutions']]
n_plots = len(leiden_keys)

fig, axes = plt.subplots(1, n_plots, figsize=(6*n_plots, 5))
if n_plots == 1:
    axes = [axes]

for i, key in enumerate(leiden_keys):
    res = BBKNN_CONFIG['leiden_resolutions'][i]
    n_clusters = adata.obs[key].nunique()
    
    sc.pl.embedding(
        adata, basis='umap_bbknn', color=key,
        ax=axes[i], show=False, legend_loc='on data',
        title=f'Leiden res={res} ({n_clusters} clusters)'
    )

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}leiden_clustering_resolutions.pdf', bbox_inches='tight', dpi=300)
plt.show()

print(f"✓ Clustering plots saved")



## 11. CellTypist Annotation Visualization (if available)



In [ ]:
print(f"\n{'='*80}")
print("STEP 11: CELLTYPIST ANNOTATION VISUALIZATION")
print(f"{'='*80}\n")

# Find CellTypist column
celltypist_col = None
for col in ['celltypist_majority_voting', 'cell_type', 'celltype', 'annotation']:
    if col in adata.obs.columns:
        celltypist_col = col
        print(f"✓ Found CellTypist annotation: '{col}'")
        break

if celltypist_col:
    # UMAP colored by CellTypist
    sc.pl.umap(
        adata, color=celltypist_col,
        frameon=False, title='CellTypist Annotation (BBKNN UMAP)',
        save='_bbknn_celltypist.pdf'
    )
    
    # Cell type composition
    celltypist_counts = adata.obs[celltypist_col].value_counts()
    print(f"\nCellTypist annotation distribution:")
    for ct, count in celltypist_counts.items():
        pct = count / adata.n_obs * 100
        print(f"  {ct}: {count:,} ({pct:.1f}%)")
else:
    print(f"⚠️  No CellTypist annotation found")



## 12. Marker Gene Expression Visualization



In [ ]:
print(f"\n{'='*80}")
print("STEP 12: MARKER GENE EXPRESSION")
print(f"{'='*80}\n")

# Define secretory markers
secretory_markers = {
    'Secretory_General': ['SCGB1A1', 'SCGB3A1', 'MUC5B'],
    'Goblet': ['MUC5AC', 'TFF3', 'SPDEF'],
    'Serous': ['LTF', 'LYZ', 'DMBT1'],
    'Club': ['SCGB1A1', 'SCGB3A2', 'CYP2F1'],
    'Basal': ['KRT5', 'TP63', 'KRT14'],
    'Stress': ['HSP90AA1', 'HSPA1A', 'HSPA1B'],
}

# Flatten and check availability
all_markers = []
for markers in secretory_markers.values():
    all_markers.extend(markers)
all_markers = list(set(all_markers))

if adata.raw is not None:
    available_markers = [g for g in all_markers if g in adata.raw.var_names]
    use_raw = True
else:
    available_markers = [g for g in all_markers if g in adata.var_names]
    use_raw = False

print(f"Marker genes: {len(available_markers)}/{len(all_markers)} available")
if len(available_markers) < len(all_markers):
    missing = set(all_markers) - set(available_markers)
    print(f"  Missing: {missing}")

# UMAP with marker expression
if available_markers:
    # Plot in batches of 6
    for i in range(0, len(available_markers), 6):
        batch_markers = available_markers[i:i+6]
        sc.pl.umap(
            adata, color=batch_markers, ncols=3,
            use_raw=use_raw, frameon=False, cmap='RdBu_r',
            save=f'_bbknn_markers_batch{i//6+1}.pdf'
        )
    print(f"✓ Marker expression plots saved")



## 13. Dotplot: Markers × Clusters



In [ ]:
print(f"\n{'='*80}")
print("STEP 13: MARKER DOTPLOT")
print(f"{'='*80}\n")

if available_markers:
    try:
        sc.pl.dotplot(
            adata, available_markers, 
            groupby='leiden_bbknn',
            use_raw=use_raw,
            dendrogram=True,
            save='_bbknn_markers_by_cluster.pdf'
        )
        print(f"✓ Dotplot saved")
    except Exception as e:
        print(f"⚠️  Dotplot failed: {e}")



## 14. Differential Expression: Find Markers per Cluster



In [ ]:
print(f"\n{'='*80}")
print("STEP 14: DIFFERENTIAL EXPRESSION - CLUSTER MARKERS")
print(f"{'='*80}\n")

print(f"Parameters:")
print(f"  Method: {MARKER_CONFIG['method']}")
print(f"  Min pct: {MARKER_CONFIG['min_pct']}")
print(f"  LogFC threshold: {MARKER_CONFIG['logfc_threshold']}")
print(f"  Use raw: {MARKER_CONFIG['use_raw'] and adata.raw is not None}")

# Check if DE already done
if 'rank_genes_groups' in adata.uns:
    print(f"\n⚠️  Previous DE results found, will overwrite")

# Run DE
try:
    use_raw_de = MARKER_CONFIG['use_raw'] and adata.raw is not None
    
    sc.tl.rank_genes_groups(
        adata,
        groupby='leiden_bbknn',
        method=MARKER_CONFIG['method'],
        use_raw=use_raw_de,
        pts=True,  # Calculate percentage expressed
        key_added='rank_genes_groups',
        n_genes=200,  # Get more genes for downstream analysis
    )
    print(f"\n✓ Differential expression completed")
    
except Exception as e:
    raise RuntimeError(f"Differential expression failed: {e}")

# Extract top markers per cluster
print(f"\nTop {MARKER_CONFIG['top_n']} markers per cluster:")

results_dict = {
    'cluster': [],
    'gene': [],
    'logfoldchanges': [],
    'pvals_adj': [],
    'pct_in_cluster': [],
    'pct_out_cluster': [],
}

for cluster in adata.obs['leiden_bbknn'].cat.categories:
    de_result = sc.get.rank_genes_groups_df(
        adata, 
        group=cluster,
        key='rank_genes_groups'
    )
    
    # Filter by thresholds
    de_filtered = de_result[
        (de_result['pvals_adj'] < 0.05) &
        (de_result['logfoldchanges'] > MARKER_CONFIG['logfc_threshold']) &
        (de_result['pct_nz_group'] > MARKER_CONFIG['min_pct'])
    ].head(MARKER_CONFIG['top_n'])
    
    print(f"\n  Cluster {cluster}: {len(de_filtered)} significant markers")
    if len(de_filtered) > 0:
        print(f"    Top 3: {', '.join(de_filtered['names'].head(3).tolist())}")
    
    # Store results
    for _, row in de_filtered.iterrows():
        results_dict['cluster'].append(cluster)
        results_dict['gene'].append(row['names'])
        results_dict['logfoldchanges'].append(row['logfoldchanges'])
        results_dict['pvals_adj'].append(row['pvals_adj'])
        results_dict['pct_in_cluster'].append(row['pct_nz_group'])
        results_dict['pct_out_cluster'].append(row['pct_nz_reference'])

# Save to DataFrame
markers_df = pd.DataFrame(results_dict)
markers_df.to_csv(f'{OUTPUT_DIR}cluster_markers_top{MARKER_CONFIG["top_n"]}.csv', index=False)
print(f"\n✓ Markers saved to cluster_markers_top{MARKER_CONFIG['top_n']}.csv")



## 15. Differential Expression: CellTypist Annotation Markers



In [ ]:
print(f"\n{'='*80}")
print("STEP 15: DIFFERENTIAL EXPRESSION - CELLTYPIST MARKERS")
print(f"{'='*80}\n")

if celltypist_col:
    print(f"Finding markers for CellTypist annotation: {celltypist_col}")
    
    try:
        use_raw_de = MARKER_CONFIG['use_raw'] and adata.raw is not None
        
        sc.tl.rank_genes_groups(
            adata,
            groupby=celltypist_col,
            method=MARKER_CONFIG['method'],
            use_raw=use_raw_de,
            pts=True,
            key_added='rank_genes_celltypist',
            n_genes=200,
        )
        print(f"✓ DE by CellTypist annotation completed")
        
        # Extract top markers
        print(f"\nTop {MARKER_CONFIG['top_n']} markers per cell type:")
        
        celltypist_results = {
            'cell_type': [],
            'gene': [],
            'logfoldchanges': [],
            'pvals_adj': [],
            'pct_in_type': [],
            'pct_out_type': [],
        }
        
        for cell_type in adata.obs[celltypist_col].unique():
            if pd.isna(cell_type):
                continue
                
            de_result = sc.get.rank_genes_groups_df(
                adata,
                group=cell_type,
                key='rank_genes_celltypist'
            )
            
            # Filter
            de_filtered = de_result[
                (de_result['pvals_adj'] < 0.05) &
                (de_result['logfoldchanges'] > MARKER_CONFIG['logfc_threshold']) &
                (de_result['pct_nz_group'] > MARKER_CONFIG['min_pct'])
            ].head(MARKER_CONFIG['top_n'])
            
            print(f"\n  {cell_type}: {len(de_filtered)} significant markers")
            if len(de_filtered) > 0:
                print(f"    Top 3: {', '.join(de_filtered['names'].head(3).tolist())}")
            
            # Store
            for _, row in de_filtered.iterrows():
                celltypist_results['cell_type'].append(cell_type)
                celltypist_results['gene'].append(row['names'])
                celltypist_results['logfoldchanges'].append(row['logfoldchanges'])
                celltypist_results['pvals_adj'].append(row['pvals_adj'])
                celltypist_results['pct_in_type'].append(row['pct_nz_group'])
                celltypist_results['pct_out_type'].append(row['pct_nz_reference'])
        
        # Save
        celltypist_markers_df = pd.DataFrame(celltypist_results)
        celltypist_markers_df.to_csv(
            f'{OUTPUT_DIR}celltypist_markers_top{MARKER_CONFIG["top_n"]}.csv',
            index=False
        )
        print(f"\n✓ CellTypist markers saved")
        
    except Exception as e:
        print(f"⚠️  CellTypist DE failed: {e}")
else:
    print(f"⚠️  Skipping (no CellTypist annotation)")



## 16. Heatmap Visualization of Top Markers



In [ ]:
print(f"\n{'='*80}")
print("STEP 16: MARKER HEATMAP VISUALIZATION")
print(f"{'='*80}\n")

# Cluster markers heatmap
try:
    sc.pl.rank_genes_groups_heatmap(
        adata,
        n_genes=10,
        groupby='leiden_bbknn',
        key='rank_genes_groups',
        use_raw=use_raw_de,
        show_gene_labels=True,
        dendrogram=True,
        swap_axes=False,
        save='_cluster_markers_heatmap.pdf'
    )
    print(f"✓ Cluster markers heatmap saved")
except Exception as e:
    print(f"⚠️  Cluster heatmap failed: {e}")

# CellTypist markers heatmap
if celltypist_col and 'rank_genes_celltypist' in adata.uns:
    try:
        sc.pl.rank_genes_groups_heatmap(
            adata,
            n_genes=10,
            groupby=celltypist_col,
            key='rank_genes_celltypist',
            use_raw=use_raw_de,
            show_gene_labels=True,
            dendrogram=True,
            swap_axes=False,
            save='_celltypist_markers_heatmap.pdf'
        )
        print(f"✓ CellTypist markers heatmap saved")
    except Exception as e:
        print(f"⚠️  CellTypist heatmap failed: {e}")



## 17. Cluster-CellTypist Correspondence Analysis



In [ ]:
print(f"\n{'='*80}")
print("STEP 17: CLUSTER-CELLTYPIST CORRESPONDENCE")
print(f"{'='*80}\n")

if celltypist_col:
    # Crosstab
    correspondence = pd.crosstab(
        adata.obs['leiden_bbknn'],
        adata.obs[celltypist_col],
        margins=True
    )
    
    print(f"Cluster × CellTypist contingency table:")
    print(correspondence)
    
    # Save
    correspondence.to_csv(f'{OUTPUT_DIR}cluster_celltypist_correspondence.csv')
    print(f"\n✓ Correspondence table saved")
    
    # Heatmap
    correspondence_prop = correspondence.div(correspondence.sum(axis=1), axis=0).iloc[:-1, :-1]
    
    plt.figure(figsize=(max(10, len(correspondence_prop.columns)), 
                        max(6, len(correspondence_prop))))
    sns.heatmap(
        correspondence_prop,
        annot=True, fmt='.2f', cmap='YlOrRd',
        cbar_kws={'label': 'Proportion'},
        xticklabels=True, yticklabels=True
    )
    plt.title('Cluster → CellTypist Correspondence')
    plt.xlabel('CellTypist Annotation')
    plt.ylabel('BBKNN Leiden Cluster')
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}cluster_celltypist_heatmap.pdf', bbox_inches='tight', dpi=300)
    plt.show()
    
    print(f"✓ Correspondence heatmap saved")
else:
    print(f"⚠️  Skipping (no CellTypist annotation)")



## 18. Cell Composition Analysis



In [ ]:
print(f"\n{'='*80}")
print("STEP 18: CELL COMPOSITION ANALYSIS")
print(f"{'='*80}\n")

# Cluster composition per batch
composition = pd.crosstab(
    adata.obs[batch_key],
    adata.obs['leiden_bbknn']
)

composition_prop = composition.div(composition.sum(axis=1), axis=0)

print(f"Cluster composition by batch:")
print(composition)

# Heatmap
plt.figure(figsize=(max(10, len(composition_prop.columns)), 
                    max(6, len(composition_prop))))
sns.heatmap(
    composition_prop,
    annot=True, fmt='.2f', cmap='YlGnBu',
    cbar_kws={'label': 'Proportion'}
)
plt.title(f'Cluster Composition by {batch_key}')
plt.xlabel('BBKNN Leiden Cluster')
plt.ylabel(batch_key)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}cluster_composition_by_batch.pdf', bbox_inches='tight', dpi=300)
plt.show()

print(f"✓ Composition analysis saved")



## 19. Save Processed Data



In [ ]:
print(f"\n{'='*80}")
print("STEP 19: SAVE PROCESSED DATA")
print(f"{'='*80}\n")

# Save h5ad
output_h5ad = f'{OUTPUT_DIR}Secretory_Lineage_bbknn_analyzed.h5ad'
adata.write_h5ad(output_h5ad, compression='gzip', compression_opts=9)
print(f"✓ Analyzed data saved: {output_h5ad}")
print(f"  Size: {os.path.getsize(output_h5ad) / 1e6:.1f} MB")



## 20. Generate Analysis Summary Report



In [ ]:
print(f"\n{'='*80}")
print("STEP 20: ANALYSIS SUMMARY REPORT")
print(f"{'='*80}\n")

# Generate comprehensive report
report_lines = []
report_lines.append("="*80)
report_lines.append("SECRETORY LINEAGE BBKNN ANALYSIS - SUMMARY REPORT")
report_lines.append("="*80)
report_lines.append(f"\nAnalysis Date: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}")
report_lines.append(f"Author: r2end")
report_lines.append(f"\nInput: {INPUT_FILE}")
report_lines.append(f"Output: {OUTPUT_DIR}")

report_lines.append(f"\n{'-'*80}")
report_lines.append("DATASET SUMMARY")
report_lines.append(f"{'-'*80}")
report_lines.append(f"Total cells: {adata.n_obs:,}")
report_lines.append(f"Total genes: {adata.n_vars:,}")
report_lines.append(f"Batch key: {batch_key}")
report_lines.append(f"Number of batches: {adata.obs[batch_key].nunique()}")

report_lines.append(f"\n{'-'*80}")
report_lines.append("BBKNN PARAMETERS")
report_lines.append(f"{'-'*80}")
for key, value in BBKNN_CONFIG.items():
    report_lines.append(f"{key}: {value}")

report_lines.append(f"\n{'-'*80}")
report_lines.append("CLUSTERING RESULTS")
report_lines.append(f"{'-'*80}")
for res in BBKNN_CONFIG['leiden_resolutions']:
    key = f'leiden_bbknn_res{res}'
    n_clusters = adata.obs[key].nunique()
    report_lines.append(f"Resolution {res}: {n_clusters} clusters")

report_lines.append(f"\nDefault clustering (leiden_bbknn):")
for cluster, count in adata.obs['leiden_bbknn'].value_counts().sort_index().items():
    pct = count / adata.n_obs * 100
    report_lines.append(f"  Cluster {cluster}: {count:,} cells ({pct:.1f}%)")

if celltypist_col:
    report_lines.append(f"\n{'-'*80}")
    report_lines.append("CELLTYPIST ANNOTATION")
    report_lines.append(f"{'-'*80}")
    for ct, count in adata.obs[celltypist_col].value_counts().items():
        pct = count / adata.n_obs * 100
        report_lines.append(f"{ct}: {count:,} cells ({pct:.1f}%)")

report_lines.append(f"\n{'-'*80}")
report_lines.append("DIFFERENTIAL EXPRESSION")
report_lines.append(f"{'-'*80}")
report_lines.append(f"Method: {MARKER_CONFIG['method']}")
report_lines.append(f"Min pct threshold: {MARKER_CONFIG['min_pct']}")
report_lines.append(f"LogFC threshold: {MARKER_CONFIG['logfc_threshold']}")
report_lines.append(f"Top N markers: {MARKER_CONFIG['top_n']}")

if len(markers_df) > 0:
    report_lines.append(f"\nTotal significant markers (cluster): {len(markers_df)}")
    report_lines.append(f"Average markers per cluster: {len(markers_df) / adata.obs['leiden_bbknn'].nunique():.1f}")

if celltypist_col and len(celltypist_markers_df) > 0:
    report_lines.append(f"Total significant markers (CellTypist): {len(celltypist_markers_df)}")
    report_lines.append(f"Average markers per cell type: {len(celltypist_markers_df) / adata.obs[celltypist_col].nunique():.1f}")

report_lines.append(f"\n{'-'*80}")
report_lines.append("OUTPUT FILES")
report_lines.append(f"{'-'*80}")
report_lines.append(f"1. {output_h5ad}")
report_lines.append(f"2. {OUTPUT_DIR}cluster_markers_top{MARKER_CONFIG['top_n']}.csv")
if celltypist_col:
    report_lines.append(f"3. {OUTPUT_DIR}celltypist_markers_top{MARKER_CONFIG['top_n']}.csv")
    report_lines.append(f"4. {OUTPUT_DIR}cluster_celltypist_correspondence.csv")
report_lines.append(f"5. Multiple PDF figures in {OUTPUT_DIR}")

report_lines.append(f"\n{'='*80}")
report_lines.append("ANALYSIS COMPLETE")
report_lines.append(f"{'='*80}\n")

report_text = "\n".join(report_lines)
print(report_text)

# Save report
with open(f'{OUTPUT_DIR}analysis_summary_report.txt', 'w') as f:
    f.write(report_text)

print(f"✓ Summary report saved: {OUTPUT_DIR}analysis_summary_report.txt")



## 21. Next Steps Recommendations



In [ ]:
print(f"\n{'='*80}")
print("RECOMMENDED NEXT STEPS")
print(f"{'='*80}\n")

print("""
Based on this BBKNN analysis, consider:

1. **Cluster Annotation Refinement**:
   - Review marker genes per cluster
   - Compare with CellTypist predictions
   - Manual curation of ambiguous clusters

2. **Trajectory Analysis**:
   - Identify differentiation trajectories
   - Pseudotime analysis with Monocle3
   - RNA velocity if spliced/unspliced data available

3. **Functional Analysis**:
   - Gene ontology enrichment per cluster
   - Pathway analysis for secretory subtypes
   - Cell-cell interaction prediction

4. **Cross-Tissue/Disease Comparison**:
   - Compare composition across conditions
   - Identify disease-associated cell states
   - Differential abundance testing

5. **Integration with Other Modalities**:
   - Spatial transcriptomics overlay
   - Protein expression validation
   - Regulatory network inference

For detailed analysis workflows, refer to:
- QUICK_REFERENCE_MEMORY.md
- Project documentation
""")

print(f"\n{'='*80}")
print("ALL ANALYSIS COMPLETE")
print(f"{'='*80}\n")

